# **Lesson_8.2**

## In this lecture

* **CNN** for image classification task - **Part I**

*This lesson cover the first part of a sample course work focuced on **Image Classification** task using Convolutional Neural Network*
###### Based on "Predictive Analytics" module by the [Code Institute](https://codeinstitute.net/)

---

## Notebook objectives
* Build, train and test a convolutional neural network (CNN) for Classification task using an image dataset, assess the model performannce and run individual prediction on random image (from the test dataset)

## Inputs

Fashion_MNIST demo dataset from Keras

## Outputs
Pre-processed dataset `../datasets/fashion_mnist_processed.npz`

## Notes and comments
*Jupyter Notebooks of lessons 8.2 and 9.1 cover sample .ipynb file of the "Image Classification" CW. Please, bear in mind you will also have to submit README.md file, describing your ML pipeline (consult CW preparation guidelines)*

---

### Import packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import os
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist  # Importing dataset
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

### Settings

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # tf will show error messages only (reduce verbosity)
sns.set_style('white')

---

### <u>Load data</u>

In [ ]:
(X_train, y_train), (X_test, y_test) = fashion_mnist.load_data()
print(X_train.shape, X_test.shape)
# The dataset is already preprocessed and split into train and test

In [ ]:
type(X_train)

In [ ]:
n_labels = len(np.unique(y_train))
n_labels

In [ ]:
np.unique(y_train)

#### Workflow for data acquisition

* Load the data first. We are using the fashion_mnist dataset from tensorflow. It has 28x28 graiscale images of clothes and we are interested in predicting which piece of close it is.
* The data are split into train and test sets (It is available from TensorFlow in this format).
* This is a demo dataset, where all images are provided in a single and standardized format and arranged in a NumPy array.
* This is useful for learning purposes. However, actual image datasets rarely have the characteristic of having all images of the same size.

In [ ]:
pointer = 15 # number of image in the dataset (remember, numbering starts from 0!)

print(f"array pointer = {pointer}")
print(f"x_train[{pointer}] shape: {X_train[pointer].shape}")
print(f"label: {y_train[pointer]}")

plt.imshow(X_train[pointer],cmap='Accent')
plt.show()

#### Fashion MNIST dataset labels

| Label | Clothing Item |
|------|---------------|
| 0 | T-shirt / Top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle Boot |

---

### <u>Data preparation</u>

Fashion MNIST dataset is already loaded as a NumPy array, there's no need to check file extensions like we **would do** for actual image files. However, we can still verify whether all elements in X_train, X_test, and Y_train are valid image data.

* Clean data.
    * Steps to Validate Images in Fashion MNIST
        * Check the data type → Images should be numeric arrays (uint8 type).
        * Check the shape of each image → All images should be 28×28 pixels.
        * Check for corrupted images (e.g., containing NaN values).

In [ ]:
def check_images(dataset, dataset_name):
    """
    Checks images for:
    * being an array
    * shape (28x28)
    * colour channel values
    * NaN values
    """
    invalid_count = 0  # Counter for invalid images
    valid_count = 0     # Counter for valid images

    for idx, image in enumerate(dataset):
        # Check if the image is a NumPy array
        if not isinstance(image, np.ndarray):
            print(f"{dataset_name} - Index {idx}: Not a valid image array")
            invalid_count += 1
            continue

        # Check shape (should be 28x28)
        if image.shape != (28, 28):
            print(f"{dataset_name} - Index {idx}: Incorrect shape {image.shape}")
            invalid_count += 1
            continue

        # Check if values are within expected range (0-255 for grayscale images)
        if not (image.dtype == np.uint8 and image.min() >= 0 and image.max() <= 255):
            print(f"{dataset_name} - Index {idx}: Invalid pixel values (Min: {image.min()}, Max: {image.max()})")
            invalid_count += 1
            continue

        # Check for NaN values
        if np.isnan(image).any():
            print(f"{dataset_name} - Index {idx}: Contains NaN values")
            invalid_count += 1
            continue

        valid_count += 1

    print(f"\n{dataset_name}: {valid_count} valid images, {invalid_count} invalid images")


In [ ]:
# Run checks on both datasets
print("Checking Images...\n")
check_images(X_train, "Train")
check_images(X_test, "Test")

* Further data splitting: from the **train** set, we split a **validation** set. We set the validation set as 20% of the train set)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
                                    X_train,
                                    y_train,
                                    test_size=0.2,
                                    random_state=0
                                    )

print("* Train set:", X_train.shape, y_train.shape)
print("* Validation set:",  X_val.shape, y_val.shape)
print("* Test set:",   X_test.shape, y_test.shape)

---

### <u>EDA</u>

* Plot label frequency distribution in train, validation and test dataset

In [ ]:
# Define class names. Fasion MNIST has 10 labels
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

In [ ]:
# Create a DataFrame for label frequency distribution
df_freq = pd.DataFrame(columns=['Set', 'Label', 'Frequency'])


In [ ]:
def count_labels(dataset, dataset_name):
    """
    Helper function to count occurrences of each label and print them
    """
    global df_freq
    unique, counts = np.unique(dataset, return_counts=True)  # Get label frequencies
    for label, frequency in zip(unique, counts):
        df_freq = pd.concat([df_freq, pd.DataFrame([{'Set': dataset_name, 'Label': class_names[label], 'Frequency': frequency}])], ignore_index=True)
        print(f"* {dataset_name} - {class_names[label]}: {frequency} images")  # Print formatted output

In [ ]:
count_labels(y_train, "Train")
count_labels(y_test, "Test")
count_labels(y_val, "Validation")

In [ ]:
# Visualize the label distribution and save image
sns.set_style("whitegrid")
plt.figure(figsize=(10, 6))
sns.barplot(data=df_freq, x='Set', y='Frequency', hue='Label')
plt.xticks(rotation=45)
plt.title("Label Frequency Distribution in Train, Validation, and Test Sets")
# plt.savefig("/workspaces/m32895-coursework-2025/outputs/label_distribution_in_sets.png", bbox_inches='tight', dpi=150)
plt.show()

At this point we can ask ourselves:
* Is the dataset <u>balanced</u>?
* Do we have enough data?
* Are we going to run **augmentation**?


#### * We need to **reshape** and **rescale** the data to make it digestible by Tensorflow

In [ ]:
# Current data shape:
X_train.shape

* When using Convolutional Neural Networks (CNNs), the Fashion MNIST dataset needs to be reshaped to include a "channel" dimension because CNNs typically expect 4D input

| Dataset Type | Required Shape for CNN |
|--------------|------------------------|
| Grayscale Images (e.g. Fashion MNIST) | (num_images, height, width, 1) |
| RGB Images | (num_images, height, width, 3) |

N.b.: we are taking these steps for scaling the data and reshaping to include the channel dimension since the data was provided in such format and is in a NumPy array format

When you get image datasets in a NumPy format, you will recheck these items, and if required, you will need to process them.
However, when dealing with real images, the preprocessing tasks are done in another way, which we will cover in the walkthrough of project 1

In [ ]:
# Reshape Fashion MNIST data for CNN
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2], 1)
X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], X_val.shape[2], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2], 1)

# Check the new shape
print(X_train.shape)  # Expected output: (48000, 28, 28, 1)


* Convert to **float** and reshape.

In [ ]:
X_train.max()

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_val = X_val.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

In [ ]:
X_train.max()

* What Happens After Scaling?

| Before Scaling (uint8) | After Scaling (float32) |
|---|---|
| Pixel values: 0 - 255 | Pixel values: 0.0 - 1.0 |
| Integer type (uint8) | Floating-point (float32)|
| Can cause instability in training | Helps stable and faster training |
* The model learns better and converges faster when inputs are scaled to [0,1].



In [ ]:
# **Convert labels to categorical format**
n_labels = 10  # Fashion MNIST has 10 classes
y_train = to_categorical(y_train, num_classes=n_labels)
y_val = to_categorical(y_val, num_classes=n_labels)
y_test = to_categorical(y_test, num_classes=n_labels)

In [ ]:
y_test.shape

### Save datasets for the future use

In [ ]:
np.savez_compressed(
    "../datasets/fashion_mnist_processed.npz",
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test
)

---

*End of Part I*
##### Reminder: do not forget to **Clear All Outputs**
### Now you can commit and push your code to **GitHub**